# Notebook 1: Data Pipeline

**Project:** Automated Equity Research Platform

This notebook runs independently from the other notebooks.

## Standalone setup

This notebook is designed to run by itself. It can use a local clone, a Google Drive folder, or a GitHub repository.

Before publishing, replace `YOUR_USERNAME` in `REPO_URL` with your actual GitHub username.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

PROJECT_NAME = "Automated_Equity_Research_Platform"
REPO_URL = "https://github.com/YOUR_USERNAME/Automated_Equity_Research_Platform.git"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

candidate_roots = [
    Path.cwd(),
    Path("/content") / PROJECT_NAME,
    Path("/content/drive/MyDrive") / PROJECT_NAME,
    Path("/content/drive/MyDrive/AI_Equity_Research_Platform"),
]

PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "pyproject.toml").exists()),
    None,
)

if PROJECT_ROOT is None:
    if "YOUR_USERNAME" in REPO_URL:
        raise RuntimeError(
            "Project files were not found. Upload the full project folder to Google Drive "
            "or replace YOUR_USERNAME in REPO_URL with your real GitHub username."
        )
    target = Path("/content") / PROJECT_NAME
    if not target.exists():
        subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
    PROJECT_ROOT = target

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e .

In [ ]:
import os

try:
    from google.colab import userdata
    for secret_name in ("SEC_USER_AGENT", "FRED_API_KEY"):
        try:
            value = userdata.get(secret_name)
            if value:
                os.environ[secret_name] = value
        except Exception:
            pass
except Exception:
    pass

print("SEC enabled:", bool(os.getenv("SEC_USER_AGENT")))
print("FRED enabled:", bool(os.getenv("FRED_API_KEY")))

Collect market data, statements, optional SEC data, optional FRED data, and save processed outputs.

In [ ]:
from equity_research import ProjectConfig
from equity_research.data import YahooFinanceClient, SECClient, FredClient
from equity_research.features import add_market_features
from equity_research.utils import save_frame, write_json
import pandas as pd

TICKER = input("Ticker: ").strip().upper() or "AAPL"

config = ProjectConfig(project_root=PROJECT_ROOT, start_date="2016-01-01")
config.ensure_directories()
yahoo = YahooFinanceClient()

bundle = yahoo.bundle(TICKER, start=config.start_date)
features = add_market_features(bundle.prices)

print("Price rows:", len(bundle.prices))
print("Income statement shape:", bundle.income_statement.shape)
print("Balance sheet shape:", bundle.balance_sheet.shape)
print("Cash flow shape:", bundle.cash_flow.shape)
display(bundle.prices.tail())

In [ ]:
sec_filings = pd.DataFrame()
sec_facts = {}
if os.getenv("SEC_USER_AGENT"):
    sec = SECClient(os.environ["SEC_USER_AGENT"])
    sec_filings = sec.recent_filings(TICKER, limit=15)
    sec_facts = sec.company_facts(TICKER)
    display(sec_filings)
else:
    print("SEC skipped. Add SEC_USER_AGENT to enable it.")

macro = pd.DataFrame()
if os.getenv("FRED_API_KEY"):
    fred = FredClient(os.environ["FRED_API_KEY"])
    series_map = {
        "FEDFUNDS": "Federal Funds Rate",
        "DGS10": "10-Year Treasury Yield",
        "CPIAUCSL": "Consumer Price Index",
        "UNRATE": "Unemployment Rate",
    }
    macro = pd.concat([fred.series(s, start=config.start_date).rename(label) for s, label in series_map.items()], axis=1)
    display(macro.tail())
else:
    print("FRED skipped. Add FRED_API_KEY to enable it.")

In [ ]:
saved = {}
saved["prices"] = save_frame(bundle.prices, config.raw_dir / f"{TICKER}_prices.parquet")
saved["features"] = save_frame(features, config.features_dir / f"{TICKER}_market_features.parquet")
saved["company_info"] = write_json(config.raw_dir / f"{TICKER}_company_info.json", bundle.info)

for name, frame in {
    "income_statement": bundle.income_statement,
    "balance_sheet": bundle.balance_sheet,
    "cash_flow": bundle.cash_flow,
    "quarterly_income": bundle.quarterly_income_statement,
    "quarterly_balance": bundle.quarterly_balance_sheet,
    "quarterly_cash_flow": bundle.quarterly_cash_flow,
    "actions": bundle.actions,
    "sec_filings": sec_filings,
    "macro": macro,
}.items():
    if not frame.empty:
        saved[name] = save_frame(frame, config.processed_dir / f"{TICKER}_{name}.csv")

saved

In [ ]:
from equity_research.visualization import price_dashboard
price_dashboard(features.tail(1250), TICKER).show()